In [4]:
import pandas as pd

columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income",
]

df = pd.read_csv(
    "../files/adult.data", header=None, names=columns, skipinitialspace=True
)

print("Dataset shape:", df.shape)
X = df.drop(columns="income")
y = df["income"]

y = (y == ">50K").astype(int)
from sklearn.model_selection import train_test_split

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

Dataset shape: (32561, 15)


In [7]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            make_column_selector(dtype_include=["object", "category", "string"]),
        )
    ],
    remainder="passthrough",
)

estimators = [
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42)),
    ("xgb", XGBClassifier(n_estimators=200, random_state=42)),
]

stack = StackingClassifier(
    estimators=estimators, final_estimator=LogisticRegression(max_iter=1000), cv=5
)

model = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", stack)])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8708736373407032


In [10]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "category", "string"]).columns

num_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

cat_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_cols),
        ("cat", cat_transformer, cat_cols),
    ]
)

model1 = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=42)),
    ]
)

model2 = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("clf", XGBClassifier(n_estimators=200, random_state=42)),
    ]
)

model3 = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

model1.fit(X_train, y_train)
model2.fit(X_train, y_train)
model3.fit(X_train, y_train)

p1 = model1.predict_proba(X_valid)[:, 1]
p2 = model2.predict_proba(X_valid)[:, 1]
p3 = model3.predict_proba(X_valid)[:, 1]


blend = p1 * 0.5 + p2 * 0.3 + p3 * 0.2

print(blend)

[0.70426684 0.00962294 0.008936   ... 0.02276171 0.47956383 0.98045911]
